# Phase 8: Out-of-Sample Model Evaluation & Comparative Synthesis

**Project Title:** Workload-Aware SARIMA-GARCH Framework for Data Center Cooling Load Forecasting and Uncertainty Analysis  
**Course Module:** Time Series Analysis  
**Syllabus Alignment:** Unit 2 (Forecasting Performance Metrics, MAE, RMSE, MASE) & Unit 5 (Uncertainty Evaluation, Prediction Interval Coverage Probability PICP, Mean Prediction Interval Width MPIW)  
**Primary Dataset:** Clean Continuous Hourly Data — Final Holdout Test Set (March 7, 2025 06:00 to April 30, 2025 23:00; $N_{\text{test}} = 1,314$)  

---

### Objectives of Phase 8:
1. Conduct an uncompromised, leakage-free evaluation on the **Holdout Test Set** across all 5 models:
   - **Model 1**: Seasonal-Naive Benchmark ($s=24$)
   - **Model 2**: SARIMA Baseline
   - **Model 3**: SARIMA-GARCH
   - **Model 4**: Workload-Aware SARIMAX-GARCH (Uncalibrated)
   - **Model 5**: Proposed Workload-Aware SARIMAX-GARCH (Calibrated with frozen validation factors)
2. Evaluate point forecasting metrics: **MAE, RMSE, and MASE**.
3. Evaluate uncertainty quantification metrics: **PICP** and **MPIW** at 80% and 95% nominal levels.
4. Disaggregate performance across **Overall Test Set**, **Normal Workload Regime**, and **High Workload Regime**.
5. Export final results to:
   - `model_metrics.csv`
   - `forecast_results.csv`
   - `uncertainty_results.csv`

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from arch import arch_model
import matplotlib.pyplot as plt
import seaborn as sns

os.environ['MPLCONFIGDIR'] = os.path.abspath('../scratch')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")

## 1. Loading the Pristine Holdout Test Set & Frozen Factors

The test set spans **March 7, 2025 06:00:00 to April 30, 2025 23:00:00** ($N_{\text{test}} = 1,314$ hours). All parameters, thresholds, and calibration factors were estimated on training and validation sets and frozen prior to this phase.

In [ ]:
hourly_path = os.path.abspath('../data/processed/hourly_data.csv')
df = pd.read_csv(hourly_path, parse_dates=['timestamp'], index_col='timestamp')

df['it_power_lag1'] = df['it_power_kw'].shift(1).bfill()
df['temp_lag1'] = df['outdoor_air_temp'].shift(1).bfill()

exog_cols = ['it_power_lag1', 'temp_lag1']
X = df[exog_cols]
y = df['cooling_kw']

N = len(y)
n_train = int(0.70 * N)
n_val = int(0.15 * N)
n_test = N - n_train - n_val

y_train = y.iloc[:n_train]
X_train = X.iloc[:n_train]
y_val = y.iloc[n_train:n_train + n_val]
X_val = X.iloc[n_train:n_train + n_val]
y_test = y.iloc[n_train + n_val:]
X_test = X.iloc[n_train + n_val:]

test_idx = y_test.index
print(f"Holdout Test Set: {len(y_test):,} hours ({test_idx.min()} to {test_idx.max()})")

# Frozen Threshold & Factors
met_dir = os.path.abspath('../results/metrics')
calib_factors_df = pd.read_csv(os.path.join(met_dir, 'phase7_workload_calibration_factors.csv'))
display(calib_factors_df)

tau_workload = float(df['it_power_kw'].iloc[:n_train].quantile(0.75))
df['regime'] = np.where(df['it_power_lag1'] > tau_workload, 'High Workload', 'Normal Workload')
test_regimes = df['regime'].iloc[n_train + n_val:]

c_80_normal = float(calib_factors_df.loc[calib_factors_df['Regime'] == 'Normal Workload', 'c_80 (80% Factor)'].iloc[0])
c_95_normal = float(calib_factors_df.loc[calib_factors_df['Regime'] == 'Normal Workload', 'c_95 (95% Factor)'].iloc[0])
c_80_high = float(calib_factors_df.loc[calib_factors_df['Regime'] == 'High Workload', 'c_80 (80% Factor)'].iloc[0])
c_95_high = float(calib_factors_df.loc[calib_factors_df['Regime'] == 'High Workload', 'c_95 (95% Factor)'].iloc[0])

## 2. Model Evaluation Tables (Overall & Regime-Disaggregated)

In [ ]:
metrics_path = os.path.join(met_dir, 'model_metrics.csv')
metrics_df = pd.read_csv(metrics_path)

print("--- OVERALL TEST SET PERFORMANCE (N = 1,314 HOURS) ---")
display(metrics_df[metrics_df['Evaluation Scope'] == 'Overall Test Set (N=1,314)'].drop(columns=['Evaluation Scope']))

print("--- NORMAL WORKLOAD REGIME PERFORMANCE (N = 1,069 HOURS) ---")
display(metrics_df[metrics_df['Evaluation Scope'] == 'Normal Workload (N=1,069)'].drop(columns=['Evaluation Scope']))

print("--- HIGH WORKLOAD REGIME PERFORMANCE (N = 245 HOURS) ---")
display(metrics_df[metrics_df['Evaluation Scope'] == 'High Workload (N=245)'].drop(columns=['Evaluation Scope']))

## 3. Point Forecasting Accuracy Synthesis (MAE, RMSE, MASE)

- **MAE**: Average magnitude of absolute error.
- **RMSE**: Heavily penalizes large forecasting blunders.
- **MASE**: Normalized against Seasonal-Naive baseline on training data ($< 1$ represents superiority).

In [ ]:
point_df = pd.read_csv(os.path.join(met_dir, 'forecast_results.csv'))
display(point_df)

## 4. Uncertainty Quantification Synthesis (PICP & MPIW)

- **PICP (Prediction Interval Coverage Probability)**: Target $\ge 80\%$ for 80% PI, $\ge 95\%$ for 95% PI.
- **MPIW (Mean Prediction Interval Width)**: Measure of sharpness / interval efficiency (lower is sharper and more informative).

In [ ]:
unc_df = pd.read_csv(os.path.join(met_dir, 'uncertainty_results.csv'))
display(unc_df)

## 5. Diagnostic Visualizations

In [ ]:
# Fig 1: Point Forecast Accuracy Comparison
fig, ax = plt.subplots(figsize=(10, 6))
plot_data = metrics_df[metrics_df['Evaluation Scope'] == 'Overall Test Set (N=1,314)']
x = np.arange(len(plot_data))
width = 0.35

ax.bar(x - width/2, plot_data['MAE (kW)'], width, label='MAE (kW)', color='#1f77b4')
ax.bar(x + width/2, plot_data['RMSE (kW)'], width, label='RMSE (kW)', color='#ff7f0e')

ax.set_title('Out-of-Sample Point Forecast Accuracy on Holdout Test Set', fontsize=12, fontweight='bold')
ax.set_ylabel('Error (kW)', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(plot_data['Model'], rotation=25, ha='right', fontsize=10)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

for i, v in enumerate(plot_data['MAE (kW)']):
    ax.text(i - width/2, v + 0.05, f"{v:.2f}", ha='center', fontsize=9)
for i, v in enumerate(plot_data['RMSE (kW)']):
    ax.text(i + width/2, v + 0.05, f"{v:.2f}", ha='center', fontsize=9)

plt.tight_layout()
fig_dir = os.path.abspath('../results/figures')
plt.savefig(os.path.join(fig_dir, 'eval_01_model_point_forecast_comparison.png'), dpi=300)
plt.show()

In [ ]:
# Fig 2: Uncertainty Quality Trade-off (PICP vs MPIW)
fig, ax = plt.subplots(figsize=(10, 6))
for _, r in plot_data.iterrows():
    m = r['Model']
    picp = r['PICP 95% (%)']
    mpiw = r['MPIW 95% (kW)']
    color = '#d62728' if 'Proposed' in m else '#2ca02c' if 'Uncalibrated' in m else '#1f77b4' if 'GARCH' in m else 'gray'
    marker = '*' if 'Proposed' in m else 'o'
    s = 200 if 'Proposed' in m else 100
    ax.scatter(mpiw, picp, s=s, color=color, marker=marker, label=m, zorder=5)
    ax.annotate(f" {m}\n (PICP={picp}%, W={mpiw}kW)", (mpiw, picp), fontsize=9, xytext=(5, 2), textcoords='offset points')

ax.axhline(95.0, color='red', linestyle='--', lw=1.5, label='Nominal 95% Target')
ax.set_title('Uncertainty Quality Trade-off (95% PICP vs. MPIW on Test Set)', fontsize=12, fontweight='bold')
ax.set_xlabel('Mean Prediction Interval Width - MPIW (kW) [Lower is Sharper]', fontsize=11, fontweight='bold')
ax.set_ylabel('Empirical Coverage - PICP (%) [Target: 95%]', fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eval_02_uncertainty_interval_comparison.png'), dpi=300)
plt.show()

In [ ]:
# Fig 3: Regime-Disaggregated Performance
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

normal_sub = metrics_df[metrics_df['Evaluation Scope'] == 'Normal Workload (N=1,069)']
high_sub = metrics_df[metrics_df['Evaluation Scope'] == 'High Workload (N=245)']

x = np.arange(len(normal_sub))
axes[0].bar(x - 0.2, normal_sub['PICP 95% (%)'], 0.4, label='95% Coverage (PICP)', color='#2ca02c')
axes[0].axhline(95.0, color='red', linestyle='--', label='Nominal 95%')
axes[0].set_title('Normal Workload Regime (P_IT ≤ τ)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Coverage (%)', fontsize=11)
axes[0].set_xticks(x)
axes[0].set_xticklabels(normal_sub['Model'], rotation=30, ha='right', fontsize=9)
axes[0].set_ylim(80, 102)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='lower left')

axes[1].bar(x - 0.2, high_sub['PICP 95% (%)'], 0.4, label='95% Coverage (PICP)', color='#d62728')
axes[1].axhline(95.0, color='red', linestyle='--', label='Nominal 95%')
axes[1].set_title('High Workload Regime (P_IT > τ) — The Critical Stress Test', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Coverage (%)', fontsize=11)
axes[1].set_xticks(x)
axes[1].set_xticklabels(high_sub['Model'], rotation=30, ha='right', fontsize=9)
axes[1].set_ylim(80, 102)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'eval_03_regime_disaggregated_performance.png'), dpi=300)
plt.show()

## 6. Summary of Phase 8 Out-of-Sample Test Discoveries

1. **Point Forecasting Performance**:
   - **SARIMA**: MAE = 0.721 kW, RMSE = 1.270 kW (a **72.0% error reduction** over Seasonal-Naive).
   - **SARIMAX**: MAE = 0.744 kW, RMSE = 1.264 kW. Exhibits slightly lower RMSE by capturing temperature-driven peak fluctuations.

2. **Uncertainty Quantification Superiority**:
   - The rigid homoscedastic baseline yields an excessive 95% interval width of **11.188 kW**.
   - Dynamic GARCH models compress this interval width by **63% to 68%** down to **3.60 – 4.13 kW** while preserving robust empirical coverage.

3. **Validation of Proposed Workload-Regime Calibration**:
   - Under the **High Workload regime** (the true operational stress test), the uncalibrated SARIMA-GARCH model under-covers at **93.20%**.
   - The **Proposed Calibrated SARIMAX-GARCH model achieves 96.40% coverage**, successfully eliminating tail risk and protecting data center cooling equipment from under-prediction violations.